In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os

os.listdir(os.path.join(path,'dataset'))


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset,DataLoader
import numpy as np
# Custom Dataset Class
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class DT(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))  # Get all image paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))  # Get all mask paths

        self.image_paths.sort()
        self.mask_paths.sort()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)


        return image, mask  # Return image-mask pair

In [ ]:
# TO DO

image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])


image_dir = os.path.join(path, "dataset", "images")
mask_dir = os.path.join(path, "dataset", "masks")




In [ ]:
dataset=DT(image_dir, mask_dir, transform=None, target_transform=None)


train_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

print(f"Training Samples: {len(dataset)}")

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp


In [ ]:
# TO DO

device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
import torch
from torch import nn

criterion = nn.CrossEntropyLoss() # for selecting specphic chanle
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    train_losses.append(train_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}")


In [ ]:
# TO DO
import random
import matplotlib.pyplot as plt
import numpy as np

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

model.eval()

test_samples = random.sample(range(len(dataset)), 5)

for idx in test_samples:
    img, mask = dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))

    pred_mask = torch.argmax(pred_mask,dim=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
